# 1.  Create a Spark session

In [9]:
# Step 2: Create Spark Session

import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("DAT535-PLUTO-Silver")
    .master("spark://spark-vm:7077")
    .config("spark.cores.max", "8")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

# Verify Spark session
print("Spark Version:", spark.version)
print("Spark Master:", spark.sparkContext.master)
print("Application Name:", spark.sparkContext.appName)

Spark Version: 3.5.0
Spark Master: spark://spark-vm:7077
Application Name: DAT535-PLUTO-Silver


# 2. Load the PLUTO CSV dataset

In [10]:
# Step 2: Load PLUTO CSV dataset

PLUTO_PATH = "/home/ubuntu/spark-project-data/bronze/pluto/pluto.csv"

pluto_raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(PLUTO_PATH)
)

# Display first 5 rows
pluto_raw_df.show(5, truncate=False)

+-------+---------+-------+---------------+-----------------+------+----------+----------------+--------+--------+----------+----------+---------+--------+-------------------------+---------+---------+---------+---------+--------+--------+-------+-------+-------+---------+---------+---------+-------+---------+---------+----------------------------+-------+--------+-------+-------+----------+----------+----------+---------+----------+---------+----------+--------+---------+--------+----------+--------+--------+---------+---------+---+--------+----------+-------+--------+----------+---------+---------+---------+----------+----------+--------+--------+--------+--------+-------+--------+---------+------+--------+----------+-------+---------+-------+------+----------+-----------+-------+------+-------+------+---------+------+-------+----------+-------+-------------+--------------------+-----------+------------+---------+-----+-------+-----------+-------+-------+-------+-------+-------------

# 3. Check dataset size and important columns

In [11]:
# Step 3: Explore PLUTO dataset

# Count total rows and columns
print("Total rows:", pluto_raw_df.count())
print("Total columns:", len(pluto_raw_df.columns))

# Display only important columns
pluto_raw_df.select(
    "BBL",
    "borough",
    "landuse",
    "bldgclass",
    "unitsres",
    "unitstotal",
    "numbldgs",
    "lotarea",
    "bldgarea",
    "yearbuilt"
).show(5, truncate=False)

Total rows: 858284
Total columns: 108
+----------+-------+-------+---------+--------+----------+--------+-------+--------+---------+
|BBL       |borough|landuse|bldgclass|unitsres|unitstotal|numbldgs|lotarea|bldgarea|yearbuilt|
+----------+-------+-------+---------+--------+----------+--------+-------+--------+---------+
|4087860042|QN     |1      |A1       |1       |1         |2       |4,000  |1,864   |1945     |
|4087860046|QN     |1      |A0       |1       |1         |1       |3,100  |960     |1945     |
|4087860051|QN     |1      |A0       |1       |1         |1       |4,000  |1,068   |1945     |
|4087980002|QN     |1      |A0       |1       |1         |2       |4,794  |960     |1945     |
|4087980003|QN     |1      |A0       |1       |1         |2       |4,520  |960     |1945     |
+----------+-------+-------+---------+--------+----------+--------+-------+--------+---------+
only showing top 5 rows



# 4. Check PLUTO BBL format

In [12]:
# Step 4: Check BBL format in PLUTO

pluto_raw_df.groupBy(
    F.when(
        F.col("BBL").isNull() |
        (F.trim(F.col("BBL")) == ""),
        "missing"
    ).when(
        F.trim(F.col("BBL")).rlike(r"^[1-5][0-9]{9}$"),
        "valid"
    ).otherwise("invalid").alias("bbl_status")
).count().show()

[Stage 11:>                                                         (0 + 4) / 4]

+----------+------+
|bbl_status| count|
+----------+------+
|     valid|858284|
+----------+------+



# 5. Clean the PLUTO BBL column 

In [13]:
# Step 5: Clean PLUTO BBL

pluto_silver_df = (
    pluto_raw_df

    # Preserve original BBL
    .withColumn(
        "bbl_raw",
        F.col("BBL")
    )

    # Create cleaned BBL
    .withColumn(
        "bbl",
        F.when(
            F.trim(F.col("BBL")).rlike(r"^[1-5][0-9]{9}$"),
            F.trim(F.col("BBL"))
        ).otherwise(F.lit(None).cast("string"))
    )

    # Create BBL quality status
    .withColumn(
        "bbl_status",
        F.when(
            F.col("BBL").isNull() |
            (F.trim(F.col("BBL")) == ""),
            "missing"
        ).when(
            F.col("bbl").isNotNull(),
            "valid"
        ).otherwise("invalid")
    )
)

# Display results
pluto_silver_df.select(
    "bbl_raw",
    "bbl",
    "bbl_status"
).show(5, truncate=False)

+----------+----------+----------+
|bbl_raw   |bbl       |bbl_status|
+----------+----------+----------+
|4087860042|4087860042|valid     |
|4087860046|4087860046|valid     |
|4087860051|4087860051|valid     |
|4087980002|4087980002|valid     |
|4087980003|4087980003|valid     |
+----------+----------+----------+
only showing top 5 rows



# 6. Check for duplicate BBLs

In [15]:
# If a single NYC 311 request matches both rows, the join could produce two rows for one request.
# One row = one NYC 311 service request.

# Step 6: Check for duplicate BBLs

pluto_silver_df.groupBy("bbl") \
    .count() \
    .filter(F.col("count") > 1) \
    .orderBy(F.desc("count")) \
    .show(20, truncate=False)

[Stage 15:=============================>                            (2 + 2) / 4]

+---+-----+
|bbl|count|
+---+-----+
+---+-----+



# 7. Explore the PLUTO borough column

In [16]:
# Step 7: Explore borough values in PLUTO

pluto_silver_df.groupBy("borough") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(20, truncate=False)

[Stage 18:>                                                         (0 + 4) / 4]

+-------+------+
|borough|count |
+-------+------+
|QN     |324483|
|BK     |276106|
|SI     |125829|
|BX     |89362 |
|MN     |42504 |
+-------+------+



# 8. Clean the PLUTO borough column

In [17]:
# Step 8: Clean PLUTO borough

pluto_silver_df = (
    pluto_silver_df

    # Preserve original borough
    .withColumn(
        "borough_raw",
        F.col("borough")
    )

    # Convert abbreviations to full names
    .withColumn(
        "pluto_borough",
        F.when(F.upper(F.trim(F.col("borough"))) == "MN", "MANHATTAN")
         .when(F.upper(F.trim(F.col("borough"))) == "BX", "BRONX")
         .when(F.upper(F.trim(F.col("borough"))) == "BK", "BROOKLYN")
         .when(F.upper(F.trim(F.col("borough"))) == "QN", "QUEENS")
         .when(F.upper(F.trim(F.col("borough"))) == "SI", "STATEN ISLAND")
         .otherwise(F.lit(None).cast("string"))
    )
)

# Display original and cleaned borough
pluto_silver_df.select(
    "borough_raw",
    "pluto_borough"
).show(10, truncate=False)

+-----------+-------------+
|borough_raw|pluto_borough|
+-----------+-------------+
|QN         |QUEENS       |
|QN         |QUEENS       |
|QN         |QUEENS       |
|QN         |QUEENS       |
|QN         |QUEENS       |
|QN         |QUEENS       |
|QN         |QUEENS       |
|QN         |QUEENS       |
|QN         |QUEENS       |
|QN         |QUEENS       |
+-----------+-------------+
only showing top 10 rows



# 9. Verify cleaned PLUTO borough values

In [18]:
# Step 9: Verify cleaned borough values

pluto_silver_df.groupBy("pluto_borough") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(10, truncate=False)

[Stage 22:>                                                         (0 + 4) / 4]

+-------------+------+
|pluto_borough|count |
+-------------+------+
|QUEENS       |324483|
|BROOKLYN     |276106|
|STATEN ISLAND|125829|
|BRONX        |89362 |
|MANHATTAN    |42504 |
+-------------+------+



# 10. Explore the PLUTO landuse column

In [20]:
# The landuse column tells us the general type of land use assigned to a property.
# For example, PLUTO uses codes to represent categories such as residential, commercial, industrial, and other uses.
# Que - Identify property types associated with a wide variety of city-service needs.

# Step 10: Explore PLUTO landuse values

pluto_silver_df.groupBy("landuse") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(20, truncate=False)

[Stage 25:>                                                         (0 + 4) / 4]

+-------+------+
|landuse|count |
+-------+------+
|1      |566810|
|2      |131460|
|4      |56272 |
|11     |24663 |
|5      |21243 |
|3      |13288 |
|8      |12055 |
|6      |9303  |
|10     |9157  |
|7      |6346  |
|9      |4775  |
|NULL   |2912  |
+-------+------+



# 11. Clean the PLUTO landuse column

In [21]:
# Step 11: Clean PLUTO landuse column

pluto_silver_df = (
    pluto_silver_df

    # Preserve original landuse
    .withColumn(
        "land_use_raw",
        F.col("landuse")
    )

    # Create cleaned landuse
    .withColumn(
        "land_use",
        F.when(
            F.trim(F.col("landuse")).isin(
                "1", "2", "3", "4", "5", "6",
                "7", "8", "9", "10", "11"
            ),
            F.trim(F.col("landuse"))
        ).otherwise(F.lit(None).cast("string"))
    )

    # Create landuse quality status
    .withColumn(
        "land_use_status",
        F.when(
            F.col("landuse").isNull() |
            (F.trim(F.col("landuse")) == ""),
            "missing"
        ).when(
            F.col("land_use").isNotNull(),
            "valid"
        ).otherwise("invalid")
    )
)

# Verify landuse cleaning
pluto_silver_df.groupBy("land_use_status").count().show()

[Stage 28:>                                                         (0 + 4) / 4]

+---------------+------+
|land_use_status| count|
+---------------+------+
|          valid|855372|
|        missing|  2912|
+---------------+------+



# 12. Explore the PLUTO bldgclass column

In [23]:
# bldgclass describes the classification of a building or property.
# Which properties have unusually high complaint activity compared with similar buildings?
# Which property types experience a wide variety of city-service needs?

# Step 12: Explore PLUTO building classes

pluto_silver_df.groupBy("bldgclass") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(20, truncate=False)


[Stage 31:>                                                         (0 + 4) / 4]

+---------+------+
|bldgclass|count |
+---------+------+
|A1       |122813|
|A5       |104550|
|B1       |85840 |
|B2       |80002 |
|C0       |71525 |
|B3       |61967 |
|A2       |43463 |
|B9       |25885 |
|A9       |18841 |
|V0       |17741 |
|C1       |16717 |
|C3       |15918 |
|S2       |14657 |
|C2       |13574 |
|A3       |8547  |
|A0       |8417  |
|C7       |7722  |
|K1       |7585  |
|K4       |6305  |
|V1       |5988  |
+---------+------+
only showing top 20 rows



# 13. Clean the PLUTO bldgclass column

In [24]:
# Step 13: Clean PLUTO building class

pluto_silver_df = (
    pluto_silver_df

    # Preserve original building class
    .withColumn(
        "building_class_raw",
        F.col("bldgclass")
    )

    # Create cleaned building class
    .withColumn(
        "building_class",
        F.when(
            F.col("bldgclass").isNull() |
            (F.trim(F.col("bldgclass")) == ""),
            F.lit(None).cast("string")
        ).otherwise(
            F.upper(F.trim(F.col("bldgclass")))
        )
    )

    # Create building class quality status
    .withColumn(
        "building_class_status",
        F.when(
            F.col("building_class").isNull(),
            "missing"
        ).otherwise("present")
    )
)

# Verify building class cleaning
pluto_silver_df.groupBy("building_class_status").count().show()

[Stage 34:>                                                         (0 + 4) / 4]

+---------------------+------+
|building_class_status| count|
+---------------------+------+
|              present|857820|
|              missing|   464|
+---------------------+------+



# 14. Explore unitsres and unitstotal

In [25]:
# Which properties have unusually high complaint activity compared with similar buildings?
# For example, a building with 100 apartments might naturally receive more complaints than a building with 2 apartments.
# Later, we can calculate complaint rates relative to the number of residential units.

# Step 14: Explore residential and total units

print("Residential Units:")
pluto_silver_df.groupBy("unitsres") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(20, truncate=False)

print("Total Units:")
pluto_silver_df.groupBy("unitstotal") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(20, truncate=False)

Residential Units:


+--------+------+
|unitsres|count |
+--------+------+
|1       |322392|
|2       |272686|
|0       |90072 |
|3       |76562 |
|4       |21703 |
|6       |14257 |
|8       |7818  |
|5       |5751  |
|10      |3673  |
|7       |2879  |
|16      |2511  |
|20      |2478  |
|9       |2211  |
|12      |1857  |
|24      |1155  |
|15      |1125  |
|18      |989   |
|11      |902   |
|14      |857   |
|21      |767   |
+--------+------+
only showing top 20 rows

Total Units:


[Stage 40:===========================================>              (3 + 1) / 4]

+----------+------+
|unitstotal|count |
+----------+------+
|1         |343689|
|2         |268576|
|3         |92259 |
|0         |42811 |
|4         |24110 |
|6         |15733 |
|5         |8693  |
|8         |8074  |
|7         |4545  |
|10        |4528  |
|9         |3267  |
|20        |2419  |
|16        |2303  |
|12        |1838  |
|11        |1445  |
|24        |1264  |
|15        |1257  |
|14        |1002  |
|13        |982   |
|17        |967   |
+----------+------+
only showing top 20 rows



# 15. Check missing and invalid unit counts

In [26]:
# Step 15: Check unit count formats

for column in ["unitsres", "unitstotal"]:

    print("Checking:", column)

    pluto_silver_df.groupBy(
        F.when(
            F.col(column).isNull() |
            (F.trim(F.col(column)) == ""),
            "missing"
        ).when(
            F.trim(F.col(column)).rlike(r"^[0-9]+$"),
            "valid_integer"
        ).when(
            F.trim(F.col(column)).rlike(r"^[0-9]{1,3}(,[0-9]{3})+$"),
            "comma_formatted"
        ).otherwise("unexpected").alias("value_format")
    ).count().show()

Checking: unitsres


+---------------+------+
|   value_format| count|
+---------------+------+
|  valid_integer|857543|
|comma_formatted|    92|
|        missing|   649|
+---------------+------+

Checking: unitstotal


[Stage 46:==============>                                           (1 + 3) / 4]

+---------------+------+
|   value_format| count|
+---------------+------+
|  valid_integer|857527|
|comma_formatted|   109|
|        missing|   648|
+---------------+------+



# 16. Clean and convert residential and total units

In [27]:
# Step 16: Clean residential and total unit counts

pluto_silver_df = (
    pluto_silver_df

    # Clean residential units
    .withColumn(
        "residential_units",
        F.regexp_replace(
            F.trim(F.col("unitsres")), ",", ""
        ).cast("long")
    )

    # Clean total units
    .withColumn(
        "total_units",
        F.regexp_replace(
            F.trim(F.col("unitstotal")), ",", ""
        ).cast("long")
    )
)

# Display examples containing commas in the original values
pluto_silver_df.filter(
    F.col("unitsres").contains(",") |
    F.col("unitstotal").contains(",")
).select(
    "unitsres",
    "residential_units",
    "unitstotal",
    "total_units"
).show(10, truncate=False)

+--------+-----------------+----------+-----------+
|unitsres|residential_units|unitstotal|total_units|
+--------+-----------------+----------+-----------+
|0       |0                |2,000     |2000       |
|1,250   |1250             |1,251     |1251       |
|1,116   |1116             |1,119     |1119       |
|3,378   |3378             |3,378     |3378       |
|2,121   |2121             |2,141     |2141       |
|2,659   |2659             |2,719     |2719       |
|1,236   |1236             |1,236     |1236       |
|1,614   |1614             |1,615     |1615       |
|1,934   |1934             |1,947     |1947       |
|1,272   |1272             |1,272     |1272       |
+--------+-----------------+----------+-----------+
only showing top 10 rows



# 17. Verify cleaned residential and total units

In [28]:
# Missing values remain NULL.
# Zero values remain zero.
# Residential units do not exceed total units. Any exceptions should be investigated.

# Step 17: Verify cleaned unit counts

pluto_silver_df.select(

    # Missing residential units
    F.sum(
        F.when(F.col("residential_units").isNull(), 1)
         .otherwise(0)
    ).alias("missing_residential_units"),

    # Missing total units
    F.sum(
        F.when(F.col("total_units").isNull(), 1)
         .otherwise(0)
    ).alias("missing_total_units"),

    # Properties with zero residential units
    F.sum(
        F.when(F.col("residential_units") == 0, 1)
         .otherwise(0)
    ).alias("zero_residential_units"),

    # Residential units greater than total units
    F.sum(
        F.when(
            F.col("residential_units") > F.col("total_units"), 1
        ).otherwise(0)
    ).alias("residential_exceeds_total")

).show()

[Stage 50:=============================>                            (2 + 2) / 4]

+-------------------------+-------------------+----------------------+-------------------------+
|missing_residential_units|missing_total_units|zero_residential_units|residential_exceeds_total|
+-------------------------+-------------------+----------------------+-------------------------+
|                      649|                648|                 90072|                        0|
+-------------------------+-------------------+----------------------+-------------------------+



# 18. Explore the PLUTO numbldgs column

In [29]:
# numbldgs represents the number of buildings on a tax lot.
# compare properties with different physical characteristics when analyzing complaint activity.

# Step 18: Explore number of buildings

pluto_silver_df.groupBy("numbldgs") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(20, truncate=False)

[Stage 53:>                                                         (0 + 4) / 4]

+--------+------+
|numbldgs|count |
+--------+------+
|1       |604211|
|2       |210777|
|0       |33633 |
|3       |5555  |
|4       |1115  |
|NULL    |951   |
|5       |415   |
|6       |284   |
|8       |160   |
|7       |154   |
|9       |101   |
|10      |97    |
|12      |75    |
|11      |67    |
|13      |56    |
|14      |38    |
|17      |31    |
|15      |29    |
|19      |22    |
|20      |21    |
+--------+------+
only showing top 20 rows



# 19. Check missing and invalid numbldgs values

In [30]:
# Step 19: Check number of buildings format

pluto_silver_df.groupBy(
    F.when(
        F.col("numbldgs").isNull() |
        (F.trim(F.col("numbldgs")) == ""),
        "missing"
    ).when(
        F.trim(F.col("numbldgs")).rlike(r"^[0-9]+$"),
        "valid_integer"
    ).when(
        F.trim(F.col("numbldgs")).rlike(r"^[0-9]{1,3}(,[0-9]{3})+$"),
        "comma_formatted"
    ).otherwise("unexpected").alias("value_format")
).count().show()

[Stage 56:=============================>                            (2 + 2) / 4]

+---------------+------+
|   value_format| count|
+---------------+------+
|  valid_integer|857332|
|comma_formatted|     1|
|        missing|   951|
+---------------+------+



# 20. Clean and convert the numbldgs column

In [31]:
# Step 20: Clean number of buildings

pluto_silver_df = (
    pluto_silver_df

    .withColumn(
        "number_of_buildings",
        F.regexp_replace(
            F.trim(F.col("numbldgs")), ",", ""
        ).cast("long")
    )
)

# Verify the comma-formatted value
pluto_silver_df.filter(
    F.col("numbldgs").contains(",")
).select(
    "numbldgs",
    "number_of_buildings"
).show(truncate=False)

[Stage 60:>                                                         (0 + 3) / 3]

+--------+-------------------+
|numbldgs|number_of_buildings|
+--------+-------------------+
|1,862   |1862               |
+--------+-------------------+



# 21. Verify the cleaned number_of_buildings column

In [32]:
# Step 21: Verify cleaned number of buildings

pluto_silver_df.select(

    # Count missing values
    F.sum(
        F.when(F.col("number_of_buildings").isNull(), 1)
         .otherwise(0)
    ).alias("missing_buildings"),

    # Count zero values
    F.sum(
        F.when(F.col("number_of_buildings") == 0, 1)
         .otherwise(0)
    ).alias("zero_buildings"),

    # Find maximum number of buildings
    F.max("number_of_buildings").alias("max_buildings"),

    # Count total records
    F.count("*").alias("total_properties")

).show()

[Stage 61:>                                                         (0 + 4) / 4]

+-----------------+--------------+-------------+----------------+
|missing_buildings|zero_buildings|max_buildings|total_properties|
+-----------------+--------------+-------------+----------------+
|              951|         33633|         1862|          858284|
+-----------------+--------------+-------------+----------------+



# 22. Explore and check lotarea and bldgarea

In [33]:
# Step 22: Check lotarea and bldgarea formats

for column in ["lotarea", "bldgarea"]:

    print("Checking:", column)

    pluto_silver_df.groupBy(
        F.when(
            F.col(column).isNull() |
            (F.trim(F.col(column)) == ""),
            "missing"
        ).when(
            F.trim(F.col(column)).rlike(r"^[0-9]+$"),
            "valid_integer"
        ).when(
            F.trim(F.col(column)).rlike(r"^[0-9]{1,3}(,[0-9]{3})+$"),
            "comma_formatted"
        ).otherwise("unexpected").alias("value_format")
    ).count().show()

Checking: lotarea


+---------------+------+
|   value_format| count|
+---------------+------+
|comma_formatted|844736|
|  valid_integer| 12592|
|        missing|   955|
|     unexpected|     1|
+---------------+------+

Checking: bldgarea


[Stage 67:>                                                         (0 + 4) / 4]

+---------------+------+
|   value_format| count|
+---------------+------+
|comma_formatted|787822|
|  valid_integer| 69956|
|        missing|   506|
+---------------+------+



# 

# 

# 

# 

# 

# 

# 

# 